In [4]:
import pandas as pd

INPUT_CSV = './resources/cnmf_factor_cluster_top_genes_200.csv'
df = pd.read_csv(INPUT_CSV)
df.head()

,Cluster 0 genes,Cluster 0 # of shared factors,Cluster 1 genes,Cluster 1 # of shared factors,Cluster 2 genes,Cluster 2 # of shared factors,Cluster 3 genes,Cluster 3 # of shared factors,Cluster 4 genes,Cluster 4 # of shared factors,...,Cluster 30 genes,Cluster 30 # of shared factors,Cluster 31 genes,Cluster 31 # of shared factors,Cluster 32 genes,Cluster 32 # of shared factors,Cluster 33 genes,Cluster 33 # of shared factors,Cluster 34 genes,Cluster 34 # of shared factors
0,FAM189A2,16,TNC,14,DPP6,12,CFI,3,ALCAM,13,...,ABCA1,2,ABLIM3,1,WDR62,34,HSPH1,6,ATP13A4,5
1,OGFRL1,14,CD44,14,CHST11,11,SNTG1,3,GALNT13,12,...,FNDC3B,2,NHS,1,C21ORF58,34,PPP1R15A,6,LINC00299,5
2,MAP3K5,14,VCL,13,MAP3K1,11,PLEKHG1,3,DNM3,12,...,PKP4,2,MXD1,1,MSH5,34,UBC,6,AQP4,5
3,ITPR2,14,SAMD4A,13,SLC24A3,11,PRKD1,3,ADGRL3,12,...,PLCB1,2,MYOF,1,SMC4,34,CCDC59,6,HIF3A,5
4,ETNPPL,14,IGFBP7,13,NRXN1,11,ETV6,3,MIR181A2HG,12,...,FAM20C,2,NAMPT,1,LMNB1,34,UBB,6,LINC01727,5


In [5]:
records = []
for c in df.columns:
    if 'genes' not in c.lower():
        continue
    gene_list = list(df[c].dropna())  # avoid NaNs
    o = {}
    o['cluster'] = c
    # Use the column header as our 'annotation_granular' fallback
    o['plain_query'] = (
        f"What might the following enriched gene list say about the type: {gene_list}"
    )
    o['contextual_query'] = (
        "The following gene list represents a gene program found in some subset of malignant cells isolated\n"
        "from an IDH-mutant astrocytoma. Please make predictions of how this gene program affects the malignant cells\n"
        "that express it — including structure, function (biological processes), metabolic state, interactions with the\n"
        "ECM and other cells. Use evidence not only from the astrocytoma literature and other relevant cancer literature,\n"
        "but also from the normal development and function of astrocytes.\n"
        "Rank predictions more highly where multiple genes in the list are known to be involved in a relevant process.\n"
        "Where multiple genes are known to be required for that process, assess whether all required genes are present and rank higher if they are.\n"
        + "Gene List: " + str(gene_list)
    )
    records.append(o)

In [6]:
records[0]

{'cluster': 'Cluster 0 genes',
 'plain_query': "What might the following enriched gene list say about the type: ['FAM189A2', 'OGFRL1', 'MAP3K5', 'ITPR2', 'ETNPPL', 'NRG3', 'CD38', 'FMN2', 'LINC01088', 'KCNN3', 'DAAM2', 'AC002429.2', 'OBI1-AS1', 'NTRK2', 'SYTL4', 'WDR49', 'ADGRV1', 'LIFR', 'AQP4', 'ID3', 'OSBPL11', 'DPP10', 'SERPINI2', 'TLR4', 'NAA11', 'MGAT4C', 'AC026316.5', 'EEPD1', 'RASSF4', 'AL392086.3', 'SLC4A4', 'EDNRB', 'SLC39A11', 'ATP1A2', 'SLCO1C1', 'AHCYL2', 'SPON1', 'SLC1A3', 'GRAMD2B', 'DTNA', 'AC012405.1', 'NKAIN3', 'NTM', 'SLC14A1', 'DCLK2', 'DCLK1', 'ID4', 'AC124854.1', 'LINC01094', 'PCDH9', 'GABBR2', 'PARD3B', 'PDE8A', 'LRIG1', 'C5ORF64', 'RNF19A', 'SPARCL1', 'AC093535.1', 'FADS2', 'PLEKHA5', 'ASTN2', 'ADAMTS9', 'AC073941.1', 'SLC24A4', 'PAPPA', 'AC068587.4', 'FARP1', 'SORL1', 'ARHGAP26', 'CADPS', 'ST3GAL6', 'ITPKB', 'GABRB1', 'FAM107A', 'MIR99AHG', 'ANK2', 'AC107223.1', 'PPP2R2B', 'LPL', 'AL589935.1', 'MRVI1', 'TNIK', 'AL160272.1', 'AC016766.1', 'RANBP3L', 'ARHGEF4', '

In [ ]:
from futurehouse_client import FutureHouseClient, JobNames
from pathlib import Path
from aviary.core import DummyEnv
import ldp
from dotenv import load_dotenv
import os
import json
import deep_copy as dc


load_dotenv(dotenv_path='../../.env')


client = FutureHouseClient(
    api_key=os.getenv("FHK")
)

records_out = dc(records)

query_types = {'queries': ['plain_query', 'contextual_query'], 'Jobs': ['JobNames.CROW', 'JobNames.FALCON']}

for r in [records[1]]:
    for q in query_types['queries']:
        for j in query_types['Jobs']:
            task_data = {
                "name": j,
                "query": r[q],
            }
        task_response = client.run_tasks_until_done(task_data)
        with open(f'output/{j}/{r['cluster']}.md', 'w') as f:
            js = json.loads(task_response[0].model_dump_json())
            out = f"## Cluster: {r['cluster']}\n\n"
            out += F"### Query: {r['contextual_query']}"
            f.write(js['formatted_answer'])

print("Finished")

KeyError: 'answer_formatted'

In [9]:
j


{'status': 'success',
 'query': "The following gene list represents a gene program found in some subset of malignant cells isolated\nfrom an IDH-mutant astrocytoma. Please make predictions of how this gene program affects the malignant cells\nthat express it — including structure, function (biological processes), metabolic state, interactions with the\nECM and other cells. Use evidence not only from the astrocytoma literature and other relevant cancer literature,\nbut also from the normal development and function of astrocytes.\nRank predictions more highly where multiple genes in the list are known to be involved in a relevant process.\nWhere multiple genes are known to be required for that process, assess whether all required genes are present and rank higher if they are.\nGene List: ['FAM189A2', 'OGFRL1', 'MAP3K5', 'ITPR2', 'ETNPPL', 'NRG3', 'CD38', 'FMN2', 'LINC01088', 'KCNN3', 'DAAM2', 'AC002429.2', 'OBI1-AS1', 'NTRK2', 'SYTL4', 'WDR49', 'ADGRV1', 'LIFR', 'AQP4', 'ID3', 'OSBPL11',